# 09b — Longer-history LightGBM: controlled training comparison

Run **05b (historical audit) → 06–09 → 09b → 10**. Select the project kernel and Restart Kernel and Run All. A verified existing 2024 cache is reused; otherwise this notebook downloads the 12 audited months and keeps only the selected basket plus provenance. No earlier model notebook is executed here.

**Question:** does adding 2024 training observations improve LightGBM when settings and evaluation populations are held fixed?

**Two declared candidates:** the original 31-leaf balanced reference from 09, and 09's Optuna-selected settings. Refit both with 2024 added to each 2025 training fold, then through December 2025. Do not run another parameter search or choose settings from Jan–Mar validation. Compare each with its corresponding 2025-only results from 09. Validation features/labels/rows remain exactly as prepared; history is extended in training only, making this a controlled comparison rather than a new live feature policy.

The original fold-specific 2025 item eligibility/retail policy is fixed for both histories, not relearned on 2024. Current lookup interpretation of historical records is provisional. Earlier outcomes have already been inspected, so these are exploratory results. No test rows or registered November outcomes are read. Both candidates choose separate F1 cutoffs on Jan–Mar validation; those metrics reuse cutoff-selection data.

Outputs are separate under `09b_longer_history_lightgbm/<timestamp>/`. Notebook 10 compares the two new candidates alongside the four models. This does not reproduce or replace the existing dashboard checkpoint automatically; its old AP/cutoff must not be assigned to these fits.

## 1. Verify Notebook 09 and its exact inputs

The saved model study supplies parameter choices and matching 2025-only controls. No legacy combined-training checkpoint is required. Verify source and preparation metadata before historical integration.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from importlib import reload, metadata
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
import pricecatcher_config as cfg
cfg = reload(cfg)
REPOSITORY_ROOT = Path(cfg.__file__).parent
SEED, TREES, N_JOBS = 42, 150, 2
FOLDS = ["fold_1", "fold_2", "fold_3"]
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

paths = sorted((cfg.PROJECT_ROOT / "09_lightgbm").glob("*/study_manifest.json"), reverse=True)
if not paths:
    raise FileNotFoundError("Complete 09_lightgbm.ipynb first; no completed study was found.")
base_path = paths[0]
base = json.loads(base_path.read_text())
base_selection = json.loads((base_path.parent / "selection.json").read_text())
if base.get("status") != "completed" or base_selection != base["selection"] or base_selection["family"] != "LightGBM":
    raise ValueError("A matching completed LightGBM study is required.")
for filename, digest in base["output_sha256"].items():
    if file_hash(base_path.parent / filename) != digest:
        raise ValueError(f"Notebook 09 output changed: {filename}.")
input_hashes = base["input_sha256"]
for relative, digest in input_hashes.items():
    if file_hash(cfg.PROJECT_ROOT / relative) != digest:
        raise ValueError(f"Prepared input changed: {relative}.")
for package in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]:
    if metadata.version(package) != base["package_versions"][package]:
        raise ValueError(f"Use Notebook 09's saved {package} version for this controlled comparison.")
reference = [r["parameters"] for r in base_selection["reference_settings"] if r["method"] == "Reference leaves=31"]
if len(reference) != 1:
    raise ValueError("Notebook 09's declared 31-leaf reference is missing.")
CANDIDATES = {"reference": reference[0], "tuned": base_selection["parameters"]}
folders = {name: cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" for name in FOLDS + ["holdout"]}
settings_by_fold = {name: json.loads((folder / "experiment_settings.json").read_text()) for name, folder in folders.items()}
settings = settings_by_fold["holdout"]
FEATURES, NUMERIC, CATEGORICAL = (settings[k] for k in ["feature_columns", "numeric_features", "categorical_features"])
features, keys = FEATURES, cfg.GROUP_KEYS + ["week_start"]
if settings["surge_threshold_pct"] != 10 or features != base_selection["feature_columns"]:
    raise ValueError("Keep the existing feature list and 10% target fixed.")
source = cfg.verify_source_snapshot()
if settings["preparation"].get("source_snapshot_sha256") != source["source_snapshot_sha256"]:
    raise ValueError("Prepared inputs and integrated source snapshot disagree.")
# Extra data/cleaning inputs are recorded separately from the common 2025 controls.
extra_paths = [base_path]
for name in folders:
    stage3 = folders[name].parent / "03_cleaned"
    extra_paths += [stage3 / "daily_prices.parquet", stage3 / "cleaning_summary.json"]
    cleaning = json.loads((stage3 / "cleaning_summary.json").read_text())
    if cleaning["preparation"] != settings_by_fold[name]["preparation"]:
        raise ValueError(f"{name}: cleaning and modelling preparation differ.")
print("2025-only LightGBM study:", base_path.parent)
print("Fixed longer-history candidate settings:", CANDIDATES)

## 2. Audited historical source and immutable cache

05b audits source identity/schema and bounded samples. That audit does not prove historical unit or geography accuracy. Full integration retains incomplete-metadata records in the source cache and reports them; they cannot be used for local training. No geography is guessed. Statistical flags remain unless an independent correction is documented.

In [ ]:
audit_path = cfg.DATA_HOME / "11_historical_audit" / "monthly_availability.csv"
if not audit_path.exists():
    raise FileNotFoundError("Run 05b_historical_data_audit.ipynb successfully first.")
audit = pd.read_csv(audit_path)
months = pd.period_range("2024-01", "2024-12", freq="M").astype(str).tolist()
if (sorted(audit.month.tolist()) != months or not audit.availability.eq("available").all()
        or not audit.schema_compatible.eq(True).all() or audit.etag.isna().any()):
    raise ValueError("The audit must identify all 12 accessible, schema-compatible 2024 files and their ETags.")
for _, row in audit.iterrows():
    if row.url != cfg.BASE_PRICE_URL.format(month=row.month) or not 12 <= row.file_bytes <= 64 * 1024 * 1024:
        raise ValueError("Audit URL or file size is outside the declared integration contract.")
source_policy = {"schema_version": 2, "months": months,
    "missing_metadata_policy": "retain source; audit and quarantine missing required local-series fields from supplementary training",
    "monthly_sources": audit.sort_values("month")[["month", "url", "etag", "file_bytes"]].to_dict(orient="records"),
    "item_map": {str(k): v for k, v in cfg.PROJECT_ITEM_MAP.items()}, "egg_30_codes": cfg.EGG_30_CODES,
    "lookup_sha256": {name: file_hash(cfg.LOOKUP_DIR / name) for name in ["lookup_item.parquet", "lookup_premise.parquet"]},
    "historical_metadata": "provisional current frozen lookup interpretation; historical units not independently verified"}
source_id = hashlib.sha256(json.dumps(source_policy, sort_keys=True).encode()).hexdigest()[:16]
source_dir = cfg.DATA_HOME / "12_longer_history" / "2024_source" / source_id
source_dir.mkdir(parents=True, exist_ok=True)
source_path = source_dir / "selected_prices_2024.parquet"
source_manifest_path = source_dir / "source_manifest.json"
metadata_audit_path = source_dir / "historical_metadata_review.csv"
print("Active run:", cfg.PROJECT_ROOT)
print("2024 cache:", source_dir)

In [ ]:
if source_manifest_path.exists():
    source_record = json.loads(source_manifest_path.read_text())
    if source_record["policy"] != source_policy or not source_path.exists() or file_hash(source_path) != source_record["selected_sha256"] or not metadata_audit_path.exists() or file_hash(metadata_audit_path) != source_record["metadata_audit_sha256"]:
        raise ValueError("2024 cache changed. Restore its matching source snapshot; do not bypass hashes.")
    print("Reusing verified selected 2024 source:", source_record["selected_rows"], "rows")
else:
    item = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_item.parquet")
    premise = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_premise.parquet")
    for table, key in [(item, "item_code"), (premise, "premise_code")]:
        if table[key].isna().any() or table[key].duplicated().any():
            raise ValueError("Lookup codes must be unique and non-missing.")
    units = item.set_index("item_code").unit.astype("string").str.strip().str.lower()
    for item_code in cfg.ALL_EGG_CODES:
        if units.get(item_code) != ("30 biji" if item_code in cfg.EGG_30_CODES else "10 biji"):
            raise ValueError("Egg units disagree with the conversion contract.")
    item = item[["item_code", "item", "unit", "item_group", "item_category"]]
    premise = premise[["premise_code", "premise", "state", "district", "premise_type"]]
    frames, downloaded, metadata_reviews = [], [], []
    for row in source_policy["monthly_sources"]:
        print("Downloading audited month:", row["month"])
        request = Request(row["url"], headers={"If-Match": row["etag"], "Accept-Encoding": "identity"})
        with urlopen(request, timeout=60) as response:
            if response.status != 200 or response.headers.get("ETag") != row["etag"]:
                raise ValueError("Source identity changed. Preserve this experiment and repeat the 2024 audit.")
            payload = response.read(int(row["file_bytes"]) + 1)
        if len(payload) != int(row["file_bytes"]):
            raise ValueError("Monthly payload differs from the audited file size; no partial-year cache is committed.")
        prices = pd.read_parquet(io.BytesIO(payload), columns=["date", "premise_code", "item_code", "price"])
        prices["item_code"] = pd.to_numeric(prices.item_code, errors="raise").astype("Int64")
        prices = prices.loc[prices.item_code.isin(cfg.SCOPE_ITEM_CODES)].copy()
        prices["premise_code"] = pd.to_numeric(prices.premise_code, errors="raise").astype("Int64")
        prices["date"] = pd.to_datetime(prices.date, errors="raise").dt.normalize()
        prices["price"] = pd.to_numeric(prices.price, errors="coerce")
        if prices.empty or prices.date.isna().any() or not prices.date.dt.to_period("M").eq(pd.Period(row["month"])).all():
            raise ValueError("Selected records must be nonempty and belong to their audited month.")
        if prices.duplicated(["date", "premise_code", "item_code"]).any():
            raise ValueError("Duplicate monthly source keys need review; no automatic deduplication.")
        prices["project_group"] = prices.item_code.map(cfg.PROJECT_ITEM_MAP)
        data = prices.merge(item, on="item_code", how="left", validate="many_to_one")
        data = data.merge(premise, on="premise_code", how="left", validate="many_to_one")
        for column in cfg.TEXT_COLS:
            data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
        # Keep source records intact; only model eligibility needs complete series identifiers.
        missing = data[cfg.GROUP_KEYS].isna()
        needs_review = missing.any(axis=1)
        if needs_review.any():
            review = data.loc[needs_review, ["date", "premise_code", "item_code"]].copy()
            review["missing_fields"] = missing.loc[needs_review].apply(
                lambda fields: ", ".join(fields.index[fields]), axis=1)
            review["lookup_status"] = np.where(review.premise_code.isin(premise.premise_code),
                "known shop code; required metadata incomplete", "shop code absent from frozen lookup")
            review["month"] = row["month"]
            metadata_reviews.append(review.groupby(["month", "premise_code", "item_code", "missing_fields", "lookup_status"],
                dropna=False).agg(source_records=("date", "size"), first_date=("date", "min"), last_date=("date", "max")).reset_index())
            print("Source rows needing metadata review:", int(needs_review.sum()), "(kept in cache; not eligible for training)")
        data.loc[data.item_code.isin(cfg.EGG_30_CODES), "price"] /= 3
        data.loc[data.item_code.isin(cfg.ALL_EGG_CODES), "unit"] = "10 biji"
        frames.append(data[cfg.FINAL_COLS])
        downloaded.append({"month": row["month"], "raw_sha256": hashlib.sha256(payload).hexdigest(),
            "etag": row["etag"], "selected_rows": len(data)})
        del prices, data, payload
    combined = pd.concat(frames, ignore_index=True)
    temporary = source_path.with_suffix(".tmp")
    combined.to_parquet(temporary, index=False)
    temporary.replace(source_path)
    metadata_review = (pd.concat(metadata_reviews, ignore_index=True) if metadata_reviews else
        pd.DataFrame(columns=["month", "premise_code", "item_code", "missing_fields", "lookup_status", "source_records", "first_date", "last_date"]))
    temporary = metadata_audit_path.with_suffix(".tmp")
    metadata_review.to_csv(temporary, index=False)
    temporary.replace(metadata_audit_path)
    source_record = {"policy": source_policy, "monthly_downloads": downloaded,
        "selected_rows": len(combined), "selected_sha256": file_hash(source_path),
        "metadata_audit_sha256": file_hash(metadata_audit_path),
        "source_rows_missing_required_metadata": int(metadata_review.source_records.sum())}
    temporary = source_manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(source_record, indent=2))
    temporary.replace(source_manifest_path)
    del frames, combined
    print("Committed the complete selected year; raw monthly files were not saved.")
print("Source rows missing required metadata:", source_record["source_rows_missing_required_metadata"])
print("Metadata evidence:", metadata_audit_path)
metadata_review = pd.read_csv(metadata_audit_path)
if not metadata_review.empty:
    display(metadata_review.head(20))

## 3. Rebuild the same training rules with an earlier start

For each fold, first reproduce the original training features/labels from that fold's cleaned daily records. Only then add 2024 observations, applying its frozen item/retail policy. Original training rows and labels must remain present. Training records stop at the fold cutoff, and target horizons cannot cross it. No next-week outcomes enter model features.

Validation inputs are deliberately held fixed. This tests adding training history, not changing inference features or estimating a fixed-shop inflation index. Cleaning attrition and training support are exported for each fold.

In [ ]:
def training_table(daily, start_month):
    """Same weekly median/coverage and exact-calendar history as Notebooks 04–05."""
    daily = daily.copy()
    daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
    weekly = daily.groupby(keys).agg(weekly_median_price=("price", "median"),
        weekly_std_price=("price", "std"), record_count=("price", "size"),
        premise_count=("premise_code", "nunique"), observed_days=("date", "nunique")).reset_index()
    weekly = weekly.loc[(weekly.week_start >= pd.Timestamp(start_month + "-01"))
        & ((weekly.week_start + pd.Timedelta(days=6)) <= training_end)]
    parts = []
    for _, series in weekly.groupby(cfg.GROUP_KEYS, sort=False):
        grid = series.set_index("week_start").sort_index()
        grid = grid.reindex(pd.date_range(grid.index.min(), grid.index.max(), freq="W-MON"))
        price = grid.weekly_median_price
        for lag in [1, 2, 4]:
            grid[f"lag_{lag}w_price"] = price.shift(lag)
        for window in [4, 8]:
            grid[f"rolling_mean_{window}w"] = price.shift(1).rolling(window, min_periods=2).mean()
            grid[f"rolling_std_{window}w"] = price.shift(1).rolling(window, min_periods=2).std()
        grid["history_count_4w"] = price.shift(1).rolling(4, min_periods=1).count()
        grid["weekly_price_change_pct"] = (price / price.shift(1) - 1) * 100
        grid["change"] = (price.shift(-1) / price - 1) * 100
        current_ok = grid.record_count.ge(cfg.MIN_RECORDS) & grid.premise_count.ge(cfg.MIN_PREMISES)
        next_ok = grid.record_count.shift(-1).ge(cfg.MIN_RECORDS) & grid.premise_count.shift(-1).ge(cfg.MIN_PREMISES)
        eligible = current_ok & next_ok & grid.lag_1w_price.notna() & grid.history_count_4w.ge(cfg.MIN_HISTORY_4W)
        grid["surge_label"] = cfg.surge_mask(grid.change, settings["surge_threshold_pct"],
            settings["preparation"].get("label_tolerance_pct", 0.0)).astype(int)
        part = grid.loc[price.notna() & eligible].copy()
        part.index.name = "week_start"
        parts.append(part.reset_index())
    result = pd.concat(parts, ignore_index=True)
    result["next_week_end"] = result.week_start + pd.Timedelta(days=13)
    result = result.loc[result.next_week_end.le(training_end)].copy()
    result["month"] = result.week_start.dt.month
    week_number = result.week_start.dt.isocalendar().week.astype(float)
    result["week_sin"] = np.sin(2 * np.pi * week_number / 52.1775)
    result["week_cos"] = np.cos(2 * np.pi * week_number / 52.1775)
    return result.sort_values(keys).reset_index(drop=True)

In [ ]:
def model_inputs(frame):
    X = frame[FEATURES].copy()
    for column in CATEGORICAL:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X


def build_pipeline(parameters):
    numeric = SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)
    preprocessing = ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ], sparse_threshold=1.0)
    model = LGBMClassifier(n_estimators=TREES, learning_rate=0.05, colsample_bytree=0.8,
                           max_depth=-1, objective="binary", verbosity=-1,
                           random_state=SEED, n_jobs=N_JOBS, **parameters)
    return Pipeline([("preprocess", preprocessing), ("model", model)])


def fit_and_score(parameters, training, evaluation):
    pipeline = build_pipeline(parameters)
    pipeline.fit(model_inputs(training), training.surge_label.astype(int))
    scores = pipeline.predict_proba(model_inputs(evaluation))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Model scores must be finite and in [0, 1].")
    return pipeline, scores


def evaluate(y, scores, predictions):
    y, scores, predictions = np.asarray(y, dtype=int), np.asarray(scores), np.asarray(predictions, dtype=int)
    if not len(y) or len(y) != len(scores) or len(y) != len(predictions) or not np.isfinite(scores).all():
        raise ValueError("Metrics require aligned nonempty arrays and finite scores.")
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()), "alerts": int(predictions.sum()),
            "average_precision": float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
            "precision": float(precision_score(y, predictions, zero_division=0)),
            "recall": float(recall_score(y, predictions, zero_division=0)) if y.sum() else np.nan,
            "f1": float(f1_score(y, predictions, zero_division=0)) if y.sum() else np.nan,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}


def baselines(frame):
    change = frame.weekly_price_change_pct.to_numpy(dtype=float)
    if not np.isfinite(change).all():
        raise ValueError("Current-change baseline requires supported previous-week prices.")
    tolerance = settings["preparation"].get("label_tolerance_pct", 0.0)
    alerts = (change >= 10) | np.isclose(change, 10, atol=tolerance, rtol=0)
    return {"Always no surge": (np.zeros(len(frame)), np.zeros(len(frame), dtype=int)),
            "Current-change rule": (change, alerts.astype(int))}

## 4. Fit both histories on matching evaluation rows

There is no extra Optuna search. The reference and tuned settings were fixed before this history experiment. Fold AP is measured at the same 2025 outcome weeks as 09; differences now include additional training data. Original 2025 screening remains fixed to isolate that change. Each fit learns imputation, encoding and class weights from its own training rows only.

The reconstruction tolerance is 1e-9 relative/absolute for ordinary features, with an absolute 1e-6 RM tolerance for rolling standard deviations to accommodate cancellation on constant-price histories. Missing patterns and all labels/identifiers must agree. Values are not rounded or replaced.

In [ ]:
earlier_source = pd.read_parquet(source_path, columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
earlier_source["date"] = pd.to_datetime(earlier_source.date)
if earlier_source.empty or not earlier_source.date.between("2024-01-01", "2024-12-31").all():
    raise ValueError("The historical cache must contain selected observations from 2024 only.")
extra_paths += [source_path, source_manifest_path, metadata_audit_path, audit_path]
additional_hashes = {str(p.resolve()): file_hash(p) for p in extra_paths}
# Recheck in the final save before publishing any completed result.
results = {variant: {"folds": [], "groups": []} for variant in CANDIDATES}
training_rows, cleaning_rows = [], []
control_folds = pd.read_csv(base_path.parent / "fold_comparison.csv")
control_validation = pd.read_csv(base_path.parent / "validation_comparison.csv")
for name in FOLDS + ["holdout"]:
    settings = settings_by_fold[name]
    training_end = pd.Timestamp(settings["train_end"])
    original = pd.read_parquet(folders[name] / "train.parquet").sort_values(keys).reset_index(drop=True)
    validation = pd.read_parquet(folders[name] / "validation.parquet").sort_values(keys).reset_index(drop=True)
    if (not validation.next_week_end.le(pd.Timestamp(settings["validation_end"])).all()
            or not (validation.week_start + pd.Timedelta(days=6)).gt(training_end).all()):
        raise ValueError(f"{name}: validation crosses chronological boundaries.")
    columns = cfg.GROUP_KEYS + ["date", "premise_code", "price"]
    stage3 = folders[name].parent / "03_cleaned"
    daily = pd.read_parquet(stage3 / "daily_prices.parquet", columns=columns,
                            filters=[("date", "<=", training_end)])
    daily["date"] = pd.to_datetime(daily.date)
    if daily.date.gt(training_end).any():
        raise ValueError("Training-only daily read contains future observations.")
    rebuilt = training_table(daily, "2025-01")
    ordinary = list(dict.fromkeys(keys + features + ["surge_label", "next_week_end"]))
    std_columns = [c for c in ordinary if c.startswith("rolling_std_")]
    exact_columns = [c for c in ordinary if c not in std_columns]
    pd.testing.assert_frame_equal(original[exact_columns], rebuilt[exact_columns], check_dtype=False,
                                  check_exact=False, rtol=1e-9, atol=1e-9)
    np.testing.assert_allclose(original[std_columns].to_numpy(dtype=float), rebuilt[std_columns].to_numpy(dtype=float),
                               rtol=0, atol=1e-6, equal_nan=True)
    earlier = earlier_source.copy()
    cleaning = json.loads((stage3 / "cleaning_summary.json").read_text())
    invalid = earlier.price.isna() | ~np.isfinite(earlier.price) | earlier.price.le(0)
    missing = earlier[cfg.GROUP_KEYS].isna().any(axis=1) & ~invalid
    borong = (earlier.premise_type.eq("Borong").fillna(False) & ~invalid & ~missing
              if settings["preparation"]["remove_borong"] else pd.Series(False, index=earlier.index))
    low = earlier.item_code.isin(cleaning["excluded_item_codes"]) & ~invalid & ~missing & ~borong
    keep = ~invalid & ~missing & ~borong & ~low
    cleaning_rows.append({"fold": name, "selected_source_rows": len(earlier), "invalid_removed": int(invalid.sum()),
                          "missing_metadata_removed": int(missing.sum()), "borong_removed": int(borong.sum()),
                          "frozen_sparse_item_rows_removed": int(low.sum()), "retained_rows": int(keep.sum())})
    combined = pd.concat([earlier.loc[keep], daily], ignore_index=True)
    if combined.duplicated(["date", "premise_code", "item_code"]).any() or combined.date.gt(training_end).any():
        raise ValueError("Combined training keys duplicate or exceed the training cutoff.")
    longer = training_table(combined, "2024-01")
    common = original.merge(longer[keys + ["surge_label"]], on=keys, how="inner", validate="one_to_one", suffixes=("", "_longer"))
    if len(common) != len(original) or not common.surge_label.eq(common.surge_label_longer).all():
        raise ValueError("Adding history lost original training rows or changed their labels.")
    if longer.surge_label.nunique() != 2:
        raise ValueError("Longer-history training needs both classes.")
    for history, frame in [("2025-only", original), ("2024-added", longer)]:
        training_rows.append({"fold": name, "history": history, "rows": len(frame),
                              "positives": int(frame.surge_label.sum()),
                              "first_week": str(frame.week_start.min().date()), "last_outcome": str(frame.next_week_end.max().date())})
    for variant, parameters in CANDIDATES.items():
        print("Fitting:", name, variant, "| training rows:", len(longer), flush=True)
        pipeline, scores = fit_and_score(parameters, longer, validation)
        metrics = evaluate(validation.surge_label, scores, scores >= 0.5)
        control_name = "Reference leaves=31" if variant == "reference" else "Optuna selected"
        if name in FOLDS:
            control = control_folds.loc[control_folds.method.eq(control_name) & control_folds.fold.eq(name)]
            if len(control) != 1:
                raise ValueError("Expected exactly one matching 2025-only fold control.")
            results[variant]["folds"].append({"method": "Longer history", "fold": name,
                "control_2025_ap": float(control.iloc[0].average_precision), **metrics})
        else:
            p, r, cutoffs = precision_recall_curve(validation.surge_label, scores)
            f1 = np.divide(2*p[:-1]*r[:-1], p[:-1]+r[:-1], out=np.zeros(len(cutoffs)), where=(p[:-1]+r[:-1])>0)
            cutoff = float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])
            control = control_validation.loc[control_validation.method.eq(control_name)]
            if len(control) != 1:
                raise ValueError("Expected one matching 2025-only validation control.")
            results[variant].update(model=pipeline, scores=scores, cutoff=cutoff, validation=validation,
                validation_rows=[{"method": "2025-only control @0.5", "cutoff": 0.5, **control.iloc[0].drop(labels=["method", "cutoff"]).to_dict()},
                  {"method": "Longer history", "cutoff": 0.5, **metrics},
                  {"method": "Longer history: apparent cutoff-selection performance", "cutoff": cutoff,
                   **evaluate(validation.surge_label, scores, scores >= cutoff)}])
            for group, positions in validation.groupby("project_group").indices.items():
                results[variant]["groups"].append({"project_group": group,
                    **evaluate(validation.iloc[positions].surge_label, scores[positions], scores[positions] >= cutoff)})
    del combined, longer, earlier, daily, rebuilt
training_comparison = pd.DataFrame(training_rows)
historical_cleaning = pd.DataFrame(cleaning_rows)
display(training_comparison)
display(historical_cleaning)

## 5. Save comparable candidates and explain the evidence

The two history variants use the same reference evaluation inputs as 09 plus separately hashed historical/cleaning sources. No historical file or original study is rewritten. Completed manifests are written last. A new execution writes a new timestamp, not a replacement dashboard checkpoint.

In [ ]:
if any(file_hash(cfg.PROJECT_ROOT / p) != h for p, h in input_hashes.items()) or any(file_hash(Path(p)) != h for p, h in additional_hashes.items()):
    raise ValueError("An upstream input changed during history comparison; no completed study is published.")
root_output = cfg.PROJECT_ROOT / "09b_longer_history_lightgbm" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
root_output.mkdir(parents=True, exist_ok=False)
settings = settings_by_fold["holdout"]
for variant, result in results.items():
    output = root_output / variant
    output.mkdir()
    folds = pd.DataFrame(result["folds"])
    fold_summary = folds.groupby("method").agg(mean_ap=("average_precision", "mean"), std_ap=("average_precision", "std"),
        mean_precision=("precision", "mean"), mean_recall=("recall", "mean"), mean_f1=("f1", "mean"))
    validation_table = pd.DataFrame(result["validation_rows"])
    groups = pd.DataFrame(result["groups"])
    for filename, table in {"fold_comparison.csv": folds, "fold_summary.csv": fold_summary.reset_index(),
        "validation_comparison.csv": validation_table, "validation_group_metrics.csv": groups,
        "training_comparison.csv": training_comparison, "historical_cleaning.csv": historical_cleaning}.items():
        table.to_csv(output / filename, index=False)
    prediction_keys = ["item_code", "state", "district", "premise_type", "project_group", "week_start", "next_week_end", "surge_label"]
    predictions = result["validation"][prediction_keys].copy()
    predictions["model_score"] = result["scores"]
    predictions["alert_at_05"] = result["scores"] >= 0.5
    predictions["alert_at_selected_cutoff"] = result["scores"] >= result["cutoff"]
    predictions.to_parquet(output / "validation_predictions.parquet", index=False)
    joblib.dump(result["model"], output / "lightgbm.joblib")
    selection = {"family": "LightGBM", "candidate_label": "Longer-history LightGBM ("+variant+")",
        "parameters": CANDIDATES[variant], "fixed_parameters": base_selection["fixed_parameters"],
        "seed": SEED, "n_jobs": N_JOBS, "trials": 0, "additional_search": False,
        "training_start": "2024-01", "training_end": settings["train_end"], "validation_end": settings["validation_end"],
        "feature_columns": features, "surge_threshold_pct": 10.0, "score_cutoff": result["cutoff"],
        "cutoff_rule": "maximum Jan–Mar validation F1; highest cutoff breaks ties",
        "comparison_method": "Longer history", "refit_on_validation": False,
        "score_interpretation": "uncalibrated model score"}
    (output / "selection.json").write_text(json.dumps(selection, indent=2))
    manifest = {"status": "completed", "study_status": "exploratory controlled history extension; test not accessed",
        "selection": selection, "input_sha256": input_hashes, "additional_input_sha256": additional_hashes,
        "parent_lightgbm_manifest_sha256": file_hash(base_path),
        "preparation": {name:s["preparation"] for name,s in settings_by_fold.items()},
        "historical_metadata_verified": False, "validation_features_extended": False,
        "package_versions": {p:metadata.version(p) for p in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]},
        "notebook_sha256": file_hash(REPOSITORY_ROOT / "09b_longer_history_lightgbm.ipynb"),
        "output_sha256": {p.name:file_hash(p) for p in output.iterdir()}}
    (output / "study_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Saved:", output)
    display(folds[["fold", "control_2025_ap", "average_precision"]])
    display(validation_table)
    display(groups)
print("Next: Run All in 10_model_comparison.ipynb. No dashboard model was replaced.")

## 6. Report interpretation

Describe this as a controlled training-history extension after the four model studies. The original 31-leaf reference and tuned setting allow separate observations about history and optimisation. Compare paired fold AP and identical validation populations, then report cleaning attrition, training support, false alerts and food-group metrics. Larger training sets alone do not establish better predictions or identify the cause of an improvement.

All candidates use original 2025 basket eligibility within each fold, even if an excluded item had more 2024 observations. Historical lookup assumptions and metadata exclusions limit coverage. Missing historical metadata is quarantined, not guessed; unresolved positive-price errors remain possible. Validation inputs are held fixed, and no prospective forecast availability or affordability benefit is demonstrated.

The two newly fitted history candidates are distinct from the older saved dashboard model. Report each fit's own metrics and cutoff. Notebook 10 can rank the six candidates by the declared fold rule, but adding scenarios after inspected outcomes remains exploratory. Neither its ranking nor apparent validation-cutoff performance is an independent final test. Any new deployment requires matching explanation/exports and a frozen future evaluation protocol.